# 代入式で繰り返しを防ぐ

代入式は、walrus（セイウチ）演算子で示される Python 3.8 の新構文で、長らく論じられてきたコードの重複に関する Python 言語の問題を解消します。

代入式は、if文の条件式のようなこれまで代入式が許されなかった箇所で変数に代入できるので便利です。
代入式の値は、walrus演算子の左側の識別子に代入される値です。

- 代入式: a := b
- 代入文: a = b

例えば、ジュース売り場で使う新鮮な果物が入った籠があるとします。

In [1]:
fresh_fruit = {
    'apple': 10,
    'banana': 8,
    'lemon': 5,
}

お客さんが来てレモネードを注文しました。籠に絞るレモンが少なくとも 1 つあることを確認しなければなりません。

レモンの個数を取得して、ゼロでないかどうか if 文でチェックします。

In [5]:
def make_lemonade(count):
  print(f'make lemonade')

def out_of_stock():
  print(f'Out of Stock!')

In [6]:
count = fresh_fruit.get('lemon', 0)
if count:
  make_lemonade(count)
else:
  out_of_stock()

make lemonade


この単純に見えるコードの問題点は、必要以上に読みにくいことです。

変数 count が if 文の最初のブロックだけ使われています。

count を if 文より前に定義すると、else ブロックを含めてそれ以降の全コードで変数 count をアクセスする必要があるように思えますが、実際にはそんなことはありません。

この、値を取得してゼロかチェックするコードパターンは、Python で非常によくあります。

`:=` ウォルラス表記で以下のように簡潔に記述できる

In [11]:
if count_walrus := fresh_fruit.get('lemon', 0):
  make_lemonade(count_walrus)
else:
  out_of_stock()

# if の外で count_walrus を使うこともできます
print(count_walrus)

make lemonade
5


これは、1行しか短くなっていませんが、count が if 文の第一ブロックにしか関係しないことが明らかなので、はるかに読みやすくなっています。

レモンの場合は、非ゼロチェックでよいことがわかります。

しかし、客がアップルサイダーを要求したら少なくとも 4個のリンゴが必要です。

これを if文の条件式で比較に使う次のようなコードにします。

In [12]:
def make_cider(count):
  print(f'make cider')

count =  fresh_fruit.get('apple', 0)
if count >= 4:
  make_cider(fresh_fruit['apple'])
else:
  out_of_stock()

make cider


この場合にも、walrus 演算子を使ってコードの明瞭さを向上できます。

In [13]:
if (count := fresh_fruit.get('apple', 0)) >= 4:
  make_cider(count)
else:
  out_of_stock()

make cider


In [14]:
# get の第二引数は、第一引数のキーが存在しなかった場合の返り値
print(fresh_fruit.get('lemon', 0))
print(fresh_fruit.get('grape', 0))

5
0


Python には、switch / case 文がないので、それに近い機能を実現するには、以下のような記述が好ましい

In [16]:
def slice_bananas(count):
  print(f'slice bananas')

def make_smoothies(pieces):
  print(f'make smoothies')

def make_cider(count):
  print(f'make cider')

In [17]:
if (count := fresh_fruit.get('banana', 0)) >= 2:
  pieces = slice_bananas(count)
  to_enjoy = make_smoothies(pieces)
elif (count := fresh_fruit.get('apple', 0)) >= 4:
  to_enjoy = make_cider(count)
elif count := fresh_fruit.get('lemon', 0):
  to_enjoy = make_lemonade(count)
else:
  to_enjoy = 'Nothing'
  print(f'Nothing')

slice bananas
make smoothies


Python には、do / while 文もない

In [18]:
print(fresh_fruit.items())

dict_items([('apple', 10), ('banana', 8), ('lemon', 5)])


In [56]:
import random

def pick_fruit():
  """果物の入荷をシミュレートする関数
    
  ランダムに果物とその数量を返す。
  在庫がない場合は空の辞書を返す。
  """
    
  # 果物の種類とそれぞれの確率（この日の入荷がない場合もある）
  fruits = ['apple', 'banana', 'lemon']
  
  # ランダムに今回入荷する果物を決定
  if random.random() < 0.8:  # 50%の確率で何かしらの果物が入荷
    fruit_count = {}
    for fruit in fruits:
      if random.random() < 0.3:  # 各果物が入荷する確率
        fruit_count[fruit] = random.randint(1, 3)
    return fruit_count
  else:
    return {}  # 入荷なし

def make_juice(fruit, count):
  """果物からジュースを作る関数
    
  1つの果物から1〜3本のジュースができる
  """
  
  # それぞれの果物から作れるジュースの本数を計算
  juice_per_fruit = random.randint(1, 2)
  total_juice = count * juice_per_fruit
    
  # ジュースの種類を示す文字列のリストを返す
  return [f"{fruit} juice" for _ in range(total_juice)]


In [49]:
bottles = []
fresh_fruit = pick_fruit()
while fresh_fruit:
  for fruit, count in fresh_fruit.items():
    batch = make_juice(fruit, count)
    bottles.extend(batch)
  fresh_fruit = pick_fruit()

print(bottles)

['lemon juice', 'lemon juice', 'lemon juice', 'lemon juice', 'lemon juice', 'lemon juice']


In [50]:
if {}:
  print('{} is true.')
else:
  print('{} is false.')

{} is false.


In [58]:
bottles = []
while fresh_fruit := pick_fruit():
  for fruit, count in fresh_fruit.items():
    batch = make_juice(fruit, count)
    bottles.extend(batch)

print(bottles)

['apple juice', 'apple juice', 'apple juice', 'apple juice', 'apple juice', 'apple juice']


一般的に、ある範囲の行で同じ式や代入を複数回繰り返している場合には、読みやすくするために代入式を検討しましょう。

## 覚えておくこと

- 代入式は、walrus 演算子 := を使って変数名への値代入と評価を 1つの式で行い、繰り返しをなくす。
- 代入式がより大きな式の一部なら、括弧で括る必要がある。
- Pyhton には switch / case 文や do / while ループはないものの、それらの機能は代入式を用いて明確に書ける。